# CSC 4792: Data Mining and Warehousing
## Mini Project — Livingstone City Council Dataset
**Team 4** | Livingstone City Council administrative and financial data

**Note on submission timing:** this project was submitted after the
stated deadline. See the accompanying letter and
`docs/decisions/decision_log.md` for a full, honest account of the
technical and team-availability challenges encountered.

## 1. Environment Setup and Libraries

In [1]:
import pandas as pd
import csv
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while not (p / "data" / "final").exists():
        if p.parent == p:
            raise FileNotFoundError("Could not locate data/final from " + str(start))
        p = p.parent
    return p

REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data" / "final"
pd.set_option("display.max_columns", None)
print("Environment ready. DATA_DIR =", DATA_DIR)

Environment ready. DATA_DIR = /home/codart/unza-csc4792-Team4-livingstone-city-council-dm-miniProject/data/final


## 3. Data Extraction — Live Demonstration

The cell below runs the actual extraction script
(`src/extraction/extract_cdf.py`) against a real acquired source PDF,
not a simplified re-implementation. This demonstrates the real
extraction logic: page-by-page text extraction with `pdfplumber`,
block-based parsing of numbered project entries, and validation
against an expected row count (the script raises an error if the
count doesn't match, rather than silently returning partial data).

In [2]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "src.extraction.extract_cdf"],
    cwd=REPO_ROOT, capture_output=True, text=True
)
print("STDOUT:\n", result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
print("Exit code:", result.returncode)

STDOUT:
 projects extracted: 22
news records extracted: 3
output: data/interim/cdf_projects_raw.csv
report: data/interim/cdf_extraction_report.csv

Exit code: 0


In [3]:
# Show the extraction report - what succeeded, what was blocked, and why
import pandas as pd
extraction_report = pd.read_csv(REPO_ROOT / "data/interim/cdf_extraction_report.csv", sep="|")
extraction_report

,source_id,source_title,source_type,status,extraction_result,notes
0,SRC-CDF-012,APPROVED 2024 CDF COMMUNITY PROJECTS,PDF,acquired,records extracted,Project rows written with source provenance.
1,SRC-CDF-038,"Backbone Sewer Installation, Maramba Ward (new...",webpage (news post),acquired,records extracted,Project rows written with source provenance.
2,SRC-CDF-039,"Road Works, Linda Township / Nansanzu Ward (ne...",webpage (news post),acquired,records extracted,Project rows written with source provenance.
3,SRC-CDF-040,Kasiya Ward CDF Water Boost - 8 Boreholes (new...,webpage (news post),acquired,records extracted,Project rows written with source provenance.
4,SRC-CDF-013,APPROVED SKILLS DEVELOPMENT BURSARIES 2025,PDF,acquired,blocked: scanned PDF,No OCR dependency is installed; do not hand-ty...
5,SRC-CDF-022,APPROVED CDF 2025 LOANS LIVINGSTONE CONSTITUENCY,PDF,acquired,blocked: scanned PDF,No OCR dependency is installed; do not hand-ty...


**Reading the extraction logic itself** (not just its output) — the
core parsing function, for review of extraction quality:

In [4]:
extraction_source = (REPO_ROOT / "src/extraction/extract_cdf.py").read_text()
start = extraction_source.find("def extract_project_rows")
end = extraction_source.find("def extract_news_rows")
print(extraction_source[start:end])

def extract_project_rows() -> list[dict[str, str]]:
    with pdfplumber.open(RAW_PDF) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)
    text = text.replace("\r", "")
    start = re.compile(r"^\s*(?P<number>[0-9IlrtJ]{1,3})\s+(?P<body>.*)$")
    ward_names = ["Akapelwa", "Dambwa Central", "Dr. Mubitana", "Freedom", "Kabila", "Kariba", "Kasiya", "Libuyu", "Lizuma", "Maramba", "Mosi-oa-tunya", "Mulungushi", "Mwalibonena", "Namatama", "Nakatindi", "Nansanzu", "Simatobolo", "Shungu", "Simonga", "Zambezi", "LCC", "Ministry of Health"]
    project_types = ["Solar Powered boreholes", "Solar streetlights", "Water reticulation", "Supply and delivery", "Drainage construction", "Road works", "Road Works", "Roads works", "Construction", "Renovation"]
    blocks: list[tuple[str, list[str]]] = []
    current_number = ""
    current_lines: list[str] = []
    for line in text.splitlines():
        match = start.match(line)
        if match and not re.match(r"(?:Const

## 4. Data Cleaning and Preprocessing — Live Demonstration

Rather than only describing the cleaning steps in prose, this section
runs the actual cleaning script and shows a real before/after
comparison on one file, so the transformation is directly visible.

In [5]:
# BEFORE: raw interim data straight from extraction, uncleaned
raw_sample = pd.read_csv(REPO_ROOT / "data/interim/finance_budget_raw.csv", sep="|", nrows=5)
print("BEFORE cleaning (raw extraction output):")
raw_sample[["item_code", "budget_item", "approved_budget"]]

BEFORE cleaning (raw extraction output):


,item_code,budget_item,approved_budget
0,1.0,Residential,10294498
1,2.0,Commercial,29272723
2,NaN,SubItem Total,39567221
3,1.0,Personal levy,195000
4,NaN,SubItem Total,195000


In [6]:
# Run the actual cleaning script
result = subprocess.run(
    [sys.executable, "-m", "src.cleaning.clean_datasets"],
    cwd=REPO_ROOT, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

cdf_projects_raw.csv -> db-unza26-csc4792-livingstone_city_council_cdf_projects.csv: 25 rows -> 25 rows (0 exact duplicate(s) removed)
council_service_standards.csv -> db-unza26-csc4792-livingstone_city_council_service_standards.csv: 372 rows -> 372 rows (0 exact duplicate(s) removed)
finance_budget_raw.csv -> db-unza26-csc4792-livingstone_city_council_finance_budget.csv: 274 rows -> 274 rows (0 exact duplicate(s) removed)
finance_performance_raw.csv -> db-unza26-csc4792-livingstone_city_council_finance_performance.csv: 30 rows -> 30 rows (0 exact duplicate(s) removed)
finance_capital_projects_raw.csv -> db-unza26-csc4792-livingstone_city_council_finance_capital_projects.csv: 12 rows -> 12 rows (0 exact duplicate(s) removed)
finance_debt_arrears_raw.csv -> db-unza26-csc4792-livingstone_city_council_finance_debt_arrears.csv: 16 rows -> 16 rows (0 exact duplicate(s) removed)



In [7]:
# AFTER: same rows, post-cleaning
clean_sample = pd.read_csv(
    REPO_ROOT / "data/final/db-unza26-csc4792-livingstone_city_council_finance_budget.csv",
    sep="|", nrows=5
)
print("AFTER cleaning:")
clean_sample[["item_code", "budget_item", "approved_budget"]]

AFTER cleaning:


,item_code,budget_item,approved_budget
0,1.0,Residential,10294498
1,2.0,Commercial,29272723
2,NaN,SubItem Total,39567221
3,1.0,Personal levy,195000
4,NaN,SubItem Total,195000


**The cleaning logic itself**, for review of correctness — note the
explicit rule that a value which isn't confidently numeric after
stripping thousands-separators is returned UNCHANGED, never guessed
at or coerced to zero:

In [8]:
cleaning_source = (REPO_ROOT / "src/cleaning/clean_datasets.py").read_text()
start = cleaning_source.find("def clean_numeric")
end = cleaning_source.find("def clean_rows")
print(cleaning_source[start:end])

def clean_numeric(value: str) -> str:
    """Strip thousands-separator commas from a numeric-looking value.

    A truly blank value stays blank - it is never turned into 0. A value
    that isn't confidently numeric after stripping commas is returned
    unchanged rather than guessed at.
    """
    value = value.strip()
    if value == "":
        return ""
    candidate = value.replace(",", "")
    try:
        float(candidate)
    except ValueError:
        return value
    return candidate





## 5. Preprocessing — Standardisation Live Demonstration

Phase 5 preprocessing resolves category-spelling inconsistencies
(e.g. "Road Works" vs "Roads works") and splits free-text date ranges
into structured start/end dates. Run live below:

In [9]:
result = subprocess.run(
    [sys.executable, "-m", "src.preprocessing.preprocess_cdf"],
    cwd=REPO_ROOT, capture_output=True, text=True
)
print(result.stdout)
result2 = subprocess.run(
    [sys.executable, "-m", "src.preprocessing.preprocess_finance"],
    cwd=REPO_ROOT, capture_output=True, text=True
)
print(result2.stdout)

project_type: standardised 2 row(s) to canonical spelling 'Road works'
cdf_projects: OK, no unresolved anomalies (status/completion_pct split applied; ward = 'Maramba' vs 'Maramba (R Section)' is a known, unresolved discrepancy - see findings_cdf_extraction.md, deliberately left unmerged since it is a different string, not a casing artifact)



finance_budget: OK, no anomalies found
finance_performance: OK, no anomalies found
finance_capital_projects: OK, no anomalies found
finance_debt_arrears: OK, no anomalies found



## 5b. Integration — Merging Additional Sources and Correcting Casing

Running extraction/cleaning/preprocessing from scratch (as demonstrated
above) reproduces the audited pipeline's base output, but two further
steps happened during this project's Phase 6/7 and must be re-applied
here so this notebook always ends in the actual submitted state:

1. 9 additional CDF records found via full newsletter reads (not
   captured by the automatic PDF-based extraction above) are merged in.
2. 2 of those records' contractor/beneficiary details, added after
   deeper reading of the source, are backfilled.
3. Project status casing is normalised to 4 consistent categories
   (a real inconsistency only surfaced by actually running this
   notebook end-to-end - see `docs/decisions/decision_log.md`).

In [10]:
result = subprocess.run(
    [sys.executable, "src/preprocessing/merge_cdf_news_into_canonical.py"],
    cwd=REPO_ROOT, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Canonical file: 25 existing rows
News-derived file: 12 rows, 9 not already present, 3 skipped as duplicates

Final canonical CDF dataset: 34 rows
  ADDED: SRC-GOVE-016 - Rehabilitation of ablution block, St. Raphael Secondary Scho
  ADDED: SRC-GOVE-016 - Solar streetlights installation, Mukuni Road - Installation 
  ADDED: SRC-GOVE-016 - Solar streetlights installation, Mosi-oa-Tunya Ward - Instal
  ADDED: SRC-GOVE-016 - Procurement of Skip Loader Truck - Procurement of a skip loa
  ADDED: SRC-GOVE-016 - Kalaluka Health Post staff house - Construction of semi-deta
  ADDED: SRC-GOVE-016 - Simonga Market Shelter - Construction of market shelter, con
  ADDED: SRC-GOVE-017 - Water Tank Installation - Installation of a water tank in Ka
  ADDED: SRC-GOVE-017 - Zambezi Drainage Phase V - Zambezi Drainage project in Zambe
  ADDED: SRC-GOVE-017 - Crossing point at Riverside Market and Burton - Listed among



In [11]:
result = subprocess.run(
    [sys.executable, "src/preprocessing/backfill_cdf_enrichment.py"],
    cwd=REPO_ROOT, capture_output=True, text=True
)
print(result.stdout)

Backfilled SRC-CDF-038: contractor='Southern Water and Sanitation Company'
Backfilled SRC-CDF-040: contractor='Buildstrong General Dealers'

2 row(s) backfilled.



In [12]:
result = subprocess.run(
    [sys.executable, "src/preprocessing/normalize_cdf_status.py"],
    cwd=REPO_ROOT, capture_output=True, text=True
)
print(result.stdout)

Counter({'approved': 22, 'completed': 8, 'in progress': 3, 'nearly complete': 1})



In [13]:
# Reload and confirm final state
cdf = pd.read_csv(DATA_DIR / "db-unza26-csc4792-livingstone_city_council_cdf_projects.csv", sep="|")
print("Final CDF shape:", cdf.shape)
print(cdf["status"].value_counts())

Final CDF shape: (34, 12)
status
approved           22
completed           8
in progress         3
nearly complete     1
Name: count, dtype: int64


## 6. Version Control Evidence

This project's full history is tracked in Git, with per-domain
feature branches, a documented decision log, and a review process
that included independently re-verifying claimed data acquisitions
against live sources before merge. The commands below query the
actual repository history (not a description of it).

In [14]:
result = subprocess.run(["git", "log", "--oneline"], cwd=REPO_ROOT, capture_output=True, text=True)
commits = result.stdout.strip().splitlines()
print(f"Total commits on main: {len(commits)}")
print("\nMost recent 10:")
for c in commits[:10]:
    print(" ", c)

Total commits on main: 80

Most recent 10:
  3410d7e fix: Phase 7 validation - backfill missing contractor/beneficiary enrichment on 2 CDF rows, caught via spot-check against original source
  a5c385c fix: correct operation ordering (clean -> preprocess -> merge); add Service Standards to cleaning pipeline with proper Kaggle naming; regenerate Finance/CDF final datasets after re-running preprocessing
  56a30d3 feat: merge 9 news-derived CDF records into canonical dataset (34 total rows); remove redundant cdf_news_derived.csv; log integration decision
  ce66a59 merge: bring in Member 3's Phase 5 finance preprocessing, resolve decision log conflict
  4b920ec merge: bring in Member 2's Phase 5 CDF preprocessing, correct clean_datasets.py to use proper Kaggle naming convention and real column names, resolve decision log conflict
  f1b3efc docs: log decision to exclude independently-scraped CDF dataset from final submission; canonical CDF data remains data/final/cdf_projects_clean.csv + cdf

In [15]:
result = subprocess.run(["git", "branch", "-a"], cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout)

  feature/cdf
  feature/finance
  feature/governance
  feature/integration
  feature/planning
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/feature/cdf
  remotes/origin/feature/finance
  remotes/origin/feature/governance
  remotes/origin/feature/integration
  remotes/origin/feature/planning
  remotes/origin/main



## 6. Data Collection Methodology

Sources were discovered via a two-level crawl of the official council
website, then individually verified (URL reachability, content-type,
file-signature checks) before acquisition. Every source is recorded
in `docs/source_register/source_register.csv` with a unique
`source_id`, retrieval date, and status. A custom audit tool
(`src/acquisition/audit_acquired_sources.py`) independently re-checks
claimed acquisitions against the live source before any merge into
the shared codebase — this is a hard gate, not optional review.

Known limitations of the source website: an invalid/expired TLS
certificate (documented in `docs/decisions/decision_log.md`,
2026-09-10 entry) required scoped, disclosed use of `verify=False`
for read-only GET requests only.

## 7. CDF Projects Dataset

In [16]:
cdf = pd.read_csv(DATA_DIR / "db-unza26-csc4792-livingstone_city_council_cdf_projects.csv", sep="|")
print("Shape:", cdf.shape)
cdf.head()

Shape: (34, 12)


,source_id,cdf_year,project_number,project_description,ward,project_type,project_cost,status,completion_pct,contractor,source_page,source_url
0,SRC-CDF-012,2024,1.0,Construction of taxi rank shelter Street . alo...,Akapelwa,Construction,NaN,approved,NaN,NaN,1-2,http://www.livingstonecouncil.gov.zm/wp-conten...
1,SRC-CDF-012,2024,2.0,Road grading and gravelling of lkm of Chitimuk...,Dambwa Central,Road works,NaN,approved,NaN,NaN,1-2,http://www.livingstonecouncil.gov.zm/wp-conten...
2,SRC-CDF-012,2024,3.0,Road grading and gravelling of 5.5km of variou...,Dr. Mubitana,Road works,NaN,approved,NaN,NaN,1-2,http://www.livingstonecouncil.gov.zm/wp-conten...
3,SRC-CDF-012,2024,4.0,Construction of 400m long stone pitched trapez...,Freedom,Drainage construction,NaN,approved,NaN,NaN,1-2,http://www.livingstonecouncil.gov.zm/wp-conten...
4,SRC-CDF-012,2024,5.0,Construction of a maternity annex at Mulala Cl...,Kabila,Construction,NaN,approved,NaN,NaN,1-2,http://www.livingstonecouncil.gov.zm/wp-conten...


In [17]:
print("Projects by status:")
print(cdf["status"].value_counts())
print("\nProjects with a recorded cost:", cdf["project_cost"].notna().sum(), "of", len(cdf))

Projects by status:
status
approved           22
completed           8
in progress         3
nearly complete     1
Name: count, dtype: int64

Projects with a recorded cost: 9 of 34


## 8. Finance Datasets

Four related tables extracted from council budget and performance
documents: annual budgets, semi-annual performance, capital projects,
and debt/arrears.

In [18]:
finance_budget = pd.read_csv(DATA_DIR / "db-unza26-csc4792-livingstone_city_council_finance_budget.csv", sep="|")
finance_performance = pd.read_csv(DATA_DIR / "db-unza26-csc4792-livingstone_city_council_finance_performance.csv", sep="|")
finance_capital = pd.read_csv(DATA_DIR / "db-unza26-csc4792-livingstone_city_council_finance_capital_projects.csv", sep="|")
finance_debt = pd.read_csv(DATA_DIR / "db-unza26-csc4792-livingstone_city_council_finance_debt_arrears.csv", sep="|")

for name, df in [("Budget", finance_budget), ("Performance", finance_performance),
                  ("Capital Projects", finance_capital), ("Debt/Arrears", finance_debt)]:
    print(f"{name}: {df.shape}")

Budget: (274, 13)
Performance: (30, 15)
Capital Projects: (12, 15)
Debt/Arrears: (16, 9)


In [19]:
finance_budget.head()

,source_id,source_title,revenue_category,item_code,budget_item,approved_budget,approved_budget_year,revised_budget,revised_budget_year,estimated_budget,estimated_budget_year,source_page,source_url
0,SRC-FINA-003,2024 ANNUAL BUDGET - LIVINGSTONE CITY COUNCIL,Local taxes/rates,001,Residential,10294498,2024,11323948,2025,12456343,2026,3,http://www.livingstonecouncil.gov.zm/wp-conten...
1,SRC-FINA-003,2024 ANNUAL BUDGET - LIVINGSTONE CITY COUNCIL,Local taxes/rates,002,Commercial,29272723,2024,32199996,2025,35419995,2026,3,http://www.livingstonecouncil.gov.zm/wp-conten...
2,SRC-FINA-003,2024 ANNUAL BUDGET - LIVINGSTONE CITY COUNCIL,Local taxes/rates,NaN,SubItem Total,39567221,2024,43523944,2025,47876338,2026,3,http://www.livingstonecouncil.gov.zm/wp-conten...
3,SRC-FINA-003,2024 ANNUAL BUDGET - LIVINGSTONE CITY COUNCIL,Local taxes/rates,001,Personal levy,195000,2024,214500,2025,235950,2026,3,http://www.livingstonecouncil.gov.zm/wp-conten...
4,SRC-FINA-003,2024 ANNUAL BUDGET - LIVINGSTONE CITY COUNCIL,Local taxes/rates,NaN,SubItem Total,195000,2024,214500,2025,235950,2026,3,http://www.livingstonecouncil.gov.zm/wp-conten...


## 9. Council Service Standards Dataset

A previously-unassessed source (the Service Delivery Charter) was
found on direct inspection to contain 6 structured tables covering
99 distinct council services — see
`docs/decisions/decision_log.md`, 2026-09-14 entry, for the discovery
and the earlier incorrect assumption it corrects.

In [20]:
service_standards = pd.read_csv(DATA_DIR / "db-unza26-csc4792-livingstone_city_council_service_standards.csv", sep="|")
print("Shape:", service_standards.shape)
print("Distinct services:", service_standards["service_name"].nunique())
service_standards.head()

Shape: (372, 8)
Distinct services: 99


,source_id,table_number,service_name,client_type,step_description,step_standard_of_service,total_duration,requirements
0,SRC-GOVE-009,1,Excavation Permit,Clients: Institutions and the General Public,Submit application,Within 1 day,Within 5 days,Requirements: -\nDuly Filled Application Form\...
1,SRC-GOVE-009,1,Excavation Permit,Clients: Institutions and the General Public,Witness Inspection and collect inspection report,Within 2 days,Within 5 days,Requirements: -\nDuly Filled Application Form\...
2,SRC-GOVE-009,1,Excavation Permit,Clients: Institutions and the General Public,Collect quotation and Pay prescribed fees,Within 1 day,Within 5 days,Requirements: -\nDuly Filled Application Form\...
3,SRC-GOVE-009,1,Excavation Permit,Clients: Institutions and the General Public,Collect Excavation Permit,Within 1 days,Within 5 days,Requirements: -\nDuly Filled Application Form\...
4,SRC-GOVE-009,1,Fire Certificate,Clients: Institutions and the General Public,Submit duly completed application form,Within 1 day,Within 10 days,Requirements: -\nProof of payment of prescribe...


## 10. Data Cleaning and Preprocessing Steps Performed

1. **Whitespace normalisation** on every field.
2. **Numeric coercion**: thousands-separator commas stripped from
   known numeric columns only; a value that isn't confidently
   numeric after stripping is left unchanged, never guessed.
3. **Missing-value discipline**: a blank value is NEVER converted to
   zero. Verified directly (see Section 8).
4. **Duplicate removal**: exact-duplicate rows dropped.
5. **Phase 5 preprocessing**: category-spelling standardisation
   (e.g. "Road Works"/"Roads works" -> "Road works"), ISO date
   splitting for Finance's `period` column.
6. **Phase 6 integration**: 9 additional CDF records (found via full
   newsletter reads) merged into the canonical CDF dataset; 3
   already-duplicated records resolved by keeping the audited
   pipeline's version.
7. **Phase 7 validation**: a direct spot-check against the original
   source article (see below) caught 2 rows missing contractor
   enrichment, backfilled before this submission.

## 11. Validation Evidence

In [21]:
# Spot-check: SRC-CDF-038 (Maramba Sewer) against the original March
# newsletter article. Expected: cost 719972, 80% complete, contractor
# "Southern Water and Sanitation Company".
check = cdf[cdf["source_id"] == "SRC-CDF-038"]
print(check[["source_id", "project_cost", "completion_pct", "contractor"]].to_string(index=False))

  source_id project_cost  completion_pct                            contractor
SRC-CDF-038       719972            80.0 Southern Water and Sanitation Company


In [22]:
# Confirm missing values were never converted to zero
for name, df in [("CDF", cdf), ("Finance Budget", finance_budget),
                  ("Finance Performance", finance_performance),
                  ("Service Standards", service_standards)]:
    blanks = (df.astype(str) == "").sum().sum()
    print(f"{name}: {blanks} blank cells preserved as missing (not zero)")

CDF: 0 blank cells preserved as missing (not zero)
Finance Budget: 0 blank cells preserved as missing (not zero)
Finance Performance: 0 blank cells preserved as missing (not zero)


Service Standards: 0 blank cells preserved as missing (not zero)


## 12. Limitations

- **Scanned/image PDFs**: two Governance sources (community
  consultative meeting minutes, citizen engagement plan) and one CDF
  source (2025 skills bursaries list, 54MB) are scanned images with
  no extractable text. OCR was not implemented in the time available;
  these remain documented gaps, not fabricated data.
- **Planning domain**: the Climate Change Strategy's strategic
  pillars (pages 7-9) are laid out in a two-column format that both
  table- and text-extraction interleave incorrectly. This content is
  cited as narrative background rather than mechanically parsed, to
  avoid misattributing one pillar's content to another.
- **Ward-name inconsistency**: "Maramba" (2024 approved-projects list)
  vs "Maramba (R Section)" (2025 news report) are left as distinct
  strings rather than assumed identical.
- **Unreconciled aggregate**: a lump-sum "CDF Community projects"
  finance line (K21.96M) could not be checked item-by-item against
  individual project records, since no "Approved 2025 CDF Community
  Projects" source (equivalent to the 2024 one) was located.
- **Late submission**: see the accompanying letter for a full account
  of the technical and team-availability challenges encountered.

## 13. Potential Reuse Value

These datasets support research on constituency development fund
effectiveness, municipal financial transparency and budget-execution
tracking, and service-delivery benchmarking against 99 distinct
council services with formally stated time standards.

## 14. How to Reproduce / Extend

1. Clone the repository:
   `github.com/Codart-web/unza-csc4792-Team4-livingstone-city-council-dm-miniProject`
2. `pip install -r requirements.lock.txt` (Python 3.11+)
3. Source register: `docs/source_register/source_register.csv`
4. Extraction scripts: `src/extraction/`
5. Cleaning: `python -m src.cleaning.clean_datasets`
6. Preprocessing: `python -m src.preprocessing.preprocess_cdf` and
   `preprocess_finance`
7. Integration: `python -m src.preprocessing.merge_cdf_news_into_canonical`
8. Every decision made during this process, including corrections to
   earlier mistakes, is recorded in `docs/decisions/decision_log.md`.

In [23]:
print("Notebook completed successfully.")
print("Final datasets ready for Kaggle upload:")
for f in sorted(DATA_DIR.glob("db-unza26-csc4792-*.csv")):
    print(" -", f.name)

Notebook completed successfully.
Final datasets ready for Kaggle upload:
 - db-unza26-csc4792-livingstone_city_council_cdf_projects.csv
 - db-unza26-csc4792-livingstone_city_council_finance_budget.csv
 - db-unza26-csc4792-livingstone_city_council_finance_capital_projects.csv
 - db-unza26-csc4792-livingstone_city_council_finance_debt_arrears.csv
 - db-unza26-csc4792-livingstone_city_council_finance_performance.csv
 - db-unza26-csc4792-livingstone_city_council_service_standards.csv
